Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
import trino
import pandas as pd

conn = trino.dbapi.connect(
    host="trino-tableau-prod.dp.navi-tech.in",
    port=443,
    user="databricks-user",
    catalog="awsdatacatalog",
    schema="default",
    http_scheme="https",
    client_tags=["databricks"],
)
cur = conn.cursor()
cur.execute("SET SESSION join_distribution_type = 'PARTITIONED'")

query = """

SELECT 
    "Final Ticket Annotated Date",
    date(CASE WHEN date_format("Final Ticket Annotated Date", '%a') = 'Sat' THEN date_add('day', 2, "Final Ticket Annotated Date") ELSE date_add('day', 1, "Final Ticket Annotated Date") END ) AS Audit_date,
    COUNT(DISTINCT CASE WHEN "Project Type" = 'SPADE Category Matching V1' THEN "Correlation Id" END) +
    COUNT(DISTINCT CASE WHEN "Project Type" = 'SPADE Template Review V1' THEN "Correlation Id" END) +
    SUM(CASE WHEN "Project Type" = 'Spade Template Tagging' THEN 1 ELSE 0 END) +
    COUNT(DISTINCT CASE WHEN "Project Type" = 'Address annotation metaaddress' THEN "Correlation Id" END) +
    COUNT(DISTINCT CASE WHEN "Project Type" = 'Address annotation clustering' THEN "Correlation Id" END) + 
    COUNT(DISTINCT CASE WHEN "Project Type" = 'Call Annotation V3' THEN "Correlation Id" END) AS "Total",
    COUNT(DISTINCT CASE WHEN "Project Type" = 'SPADE Category Matching V1' THEN "Correlation Id" END) AS "SPADE Category Matching V1",
    COUNT(DISTINCT CASE WHEN "Project Type" = 'SPADE Template Review V1' THEN "Correlation Id" END) AS "SPADE Template Review V1",
    SUM(CASE WHEN "Project Type" = 'Spade Template Tagging' THEN 1 ELSE 0 END) AS "Spade Template Tagging",
    COUNT(DISTINCT CASE WHEN "Project Type" = 'Address annotation metaaddress' THEN "Correlation Id" END) AS "Address annotation metaaddress",
    COUNT(DISTINCT CASE WHEN "Project Type" = 'Address annotation clustering' THEN "Correlation Id" END) AS "Address annotation clustering",
    COUNT(DISTINCT CASE WHEN "Project Type" = 'Call Annotation V3' THEN "Correlation Id" END) AS "Call Annotation V3"
FROM (
-- Step 1: Base combined data (your main view)
WITH combined_data AS 
--Update base query from here 
(
select * from
(
with base_table as 
(
select a.* ,
b.td "Time Taken (Seconds)" ,
c.average_time "Avg Project Time", 
c.median_time "Median Project Time", 
project_wise_daily_ticket_counts, 
project_wise_daily_time,
-- case when "Project Type" in ('Spade Template Tagging','SPADE Template Review V1','Suspicious feedback tagging','Address annotation clustering','SPADE Category Matching V1','AISHA HRC DA Tagging','Address Entity Tagging','SPADE ADF Review','Pincode_qc_v1') then round((project_wise_daily_ticket_counts/project_wise_daily_time)*c.average_time , 2) 
--      else round((project_wise_daily_ticket_counts/project_wise_daily_time)*c.median_time , 2)  
--      end "Project level Agent Productivity"
round((project_wise_daily_ticket_counts/project_wise_daily_time)*c.average_time , 2) "Project level Agent Productivity"
from 
    (
    select 
    annotated_at "Annotation Date"
    ,concat(correlation_id , '__' , cast(set as varchar)) "Correlation Id"
    ,ticket_id "Ticket Id"
    ,project_type "Project Type"
    ,u.name "Agent Name"
    ,Option_type "Option Type"
    ,type "Type"
    ,u.email_id
    from corp_analytics.data_annotation_projects_p1_tbl p1
    left join mjolnir_service.users u on u.email_id = p1.email_id
    where project_type not in  ('App Tagging' ,'Address Tagging') 
    union all
    (select 
    annotated_at "Annotation Date"
    ,correlation_id "Correlation Id"
    ,ticket_id "Ticket Id"
    ,'Address annotation metaaddress' "Project Type"
    ,u.name "Agent Name"
    ,option_type "Option Type"
    ,type "Type"
    ,u.email_id
    from 
    corp_analytics_databricks.data_annotation_meta_address ma
    left join mjolnir_service.users u on u.email_id = ma.email_id )
    union all
    (select 
    annotated_at "Annotation Date"
    ,correlation_id "Correlation Id"
    ,ticket_id "Ticket Id"
    ,'Address annotation clustering' "Project Type"
    ,u.name "Agent Name"
    ,Option_type "Option Type"
    ,type "Type"
    ,u.email_id
    from 
    corp_analytics.data_annotation_address_clustering_tbl ma
    left join mjolnir_service.users u on u.email_id = ma.email_id )
    union all
    (select 
--    cast((replace((split_part("Annotation Date",'.',1)),'T',' ')) as timestamp) +interval '330' minute "Annotation Date"
    cast("Annotation Date" as Timestamp) "Annotation Date"
    ,"Correlation Id"
    ,"Ticket Id"
    ,"Project Type"
    ,"Agent Name"
    ,"Option Type"
    ,"Type"
    ,email_id
    from corp_analytics.data_annotation_spade_template_tagging_v3_tbl)
   union all
   (select 
   annotated_at "Annotation Date"
   ,correlation_id "Correlation Id"
   ,ticket_id "Ticket Id"
   ,'SPADE Category Matching V1' "Project Type"
   ,annotated_by "Agent Name"
   ,option_type "Option Type"
   ,type "Type"
   ,email_id
   from 
   corp_analytics.data_annotation_spade_category_v3_tbl ma)
    union all
    (select 
    cast(annotated_at as timestamp) "Annotation Date"
    ,correlation_id "Correlation Id"
    ,ticket_id "Ticket Id"
    ,use_case_type "Project Type"
    ,u.name "Agent Name"
    ,option_type "Option Type"
    ,type "Type"
    ,u.email_id
    from 
    corp_analytics.data_annotation_call_annotation_v3_tbl ma
    left join mjolnir_service.users u on u.email_id = ma.email_id)
--    union all
--    (select 
--    cast((replace((split_part(ar.annotated_at,'.',1)),'T',' ')) as timestamp) +interval '330' minute "Annotation Date"
--    ,json_extract_scalar(ad.metadata,'$.correlation_id') "Correlation Id"
--    ,ad.ticket_id "Ticket Id"
--    ,json_extract_scalar(ad.metadata,'$.use_case_type') "Project Type"
--    ,u.name "Agent Name"
--    ,'YTD' "Option Type"
--    ,'YTD' "Type"
--    ,u.email_id
--    from annotation.annotation_data ad 
--    left join annotation.annotation_results ar on ad.id = ar.data_id
--    left join mjolnir_service.users u on ar.annotated_by = u.account_id 
--    where template_id in  (select "Template id" from corp_analytics_databricks.da_temporary_projects) and  ar.annotated_at is not null)
) a
left join 
    (select ticket_id , td from corp_analytics_databricks.data_annotation_online_time_tbl) b 
    on a."Ticket Id" = b.ticket_id
left join 
   (
    select date(starting_t) annotation_date, 
    title ,  round(sum(td)/count(td),2) average_time , approx_percentile(td,0.5) median_time
    from corp_analytics_databricks.data_annotation_online_time_tbl
    group by 1,2
    order by 1 desc,2
   ) c on date(a."Annotation Date") = c.annotation_date and a."Project Type" = c.title
left join 
   (
    select 
    date(starting_t) annotation_date,
    email_id, title,
    count(distinct ticket_id) project_wise_daily_ticket_counts,
    sum(td) project_wise_daily_time
    from corp_analytics_databricks.data_annotation_online_time_tbl
    group by 1,2,3
   ) d on date(a."Annotation Date") = d.annotation_date and a.email_id = d.email_id and a."Project Type" = d.title
where 
b.td is not null and a."Annotation Date" is not null and a."Ticket Id" is not null and a."Agent Name" is not null  
)
select 
bt.* 
from base_table bt
)),
--Update base query till here 
--Step 2: Correlation counts
correlation_ticket_counts AS (
    SELECT
        "Annotation Date",
        "Correlation Id",
        "Ticket Id",
        "Project Type",
        "Agent Name",
        "Option Type",
        "Type" AS "Bucket Type",
        email_id,
        "Time Taken (Seconds)",
        "Avg Project Time",
        "Median Project Time",
        project_wise_daily_ticket_counts, 
        project_wise_daily_time,
        "Project level Agent Productivity",
        COUNT(*) OVER (PARTITION BY "Correlation Id", "Project Type") AS Ticket_ID_Support,
        cast(MAX("Annotation Date") OVER (PARTITION BY "Correlation Id", "Project Type") as date) AS "Final Ticket Annotated Date"
    FROM combined_data
),
base_date as (
SELECT * FROM correlation_ticket_counts
Order by 2 desc,
1 desc
)
SELECT *
FROM base_date
WHERE "Final Ticket Annotated Date" BETWEEN current_date - interval '65' day AND current_date - interval '1' day
AND (
     ("Project Type" = 'SPADE Category Matching V1' AND Ticket_ID_Support >= 3 AND "Bucket Type" NOT IN ('Others'))
     OR
     ("Project Type" = 'SPADE Template Review V1' AND Ticket_ID_Support >= 5 AND "Bucket Type" NOT IN ('Others', 'Three -'))
     OR
     ("Project Type" = 'Spade Template Tagging')
      OR
     ("Project Type" = 'Address annotation metaaddress' AND Ticket_ID_Support >= 3 AND "Bucket Type" NOT IN ('Others'))
     OR
     ("Project Type" = 'Address annotation clustering' AND Ticket_ID_Support >= 3 AND "Bucket Type" NOT IN ('Others'))
     OR
     ("Project Type" = 'Call Annotation V3' AND Ticket_ID_Support >= 3 AND "Bucket Type" NOT IN ('Others'))
    
    )
    
ORDER BY "Final Ticket Annotated Date" DESC
)
group by 1
Order by "Final Ticket Annotated Date" ASC

"""

df = pd.read_sql(query, conn)
if not df.empty:
    print(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1jCq2HQQn_xkFNDlCFfnPvfW6VLmpJVBIFFcAsEfQeoI/edit?pli=1&gid=28816050#gid=28816050"
worksheet_name = "Annotated Tickets"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

# Combine dataframes side by side
combined_df = pd.concat([df], ignore_index=True) 

# Push the combined dataframe to Google Sheets
gs_reader.set_df_in_sheets(
    worksheet_name,
    combined_df,
    include_headers=True,
    row=1,
    col=1
)